In [ ]:
import pandas as pd
from pymongo import MongoClient
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv()

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
MONGO_URI = os.getenv("MONGO_URI")

# 1. เชื่อมต่อ
client = MongoClient(MONGO_URI)
db = client["my_project"]
collection = db["traffic_clean"]


# RAW_DIR = PROJECT_ROOT / "data" / "raw" / "traffy-fondue"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "traffy-fondue"
# 2. อ่านไฟล์ CSV
df = pd.read_csv(PROCESSED_DIR/'traffy_fondue_bangkok_processed.csv')



# 3. ดูเฉพาะ ticket_id ที่จะ insert
csv_ids = df["ticket_id"].astype(str).tolist()

print(f"📌 CSV มีทั้งหมด {len(csv_ids)} แถว")

# 4. เช็คว่ามี ticket_id ไหนที่อยู่ใน DB อยู่แล้วบ้าง
existing = collection.find(
    {"ticket_id": {"$in": csv_ids}},
    {"ticket_id": 1, "_id": 0}
)

existing_ids = {item["ticket_id"] for item in existing}

print(f"🔍 พบข้อมูลซ้ำใน MongoDB จำนวน {len(existing_ids)} รายการ")

# 5. กรองข้อมูลออก: เหลือเฉพาะ ticket_id ที่ไม่ซ้ำเท่านั้น
df_unique = df[~df["ticket_id"].astype(str).isin(existing_ids)]

print(f"✨ จะทำการ insert เฉพาะข้อมูลใหม่จำนวน {len(df_unique)} รายการ")

# 6. แปลงเป็น dict แล้ว insert
data_dict = df_unique.to_dict("records")

if len(data_dict) > 0:
    try:
        collection.insert_many(data_dict)
        print("✅ Insert สำเร็จ!")
    except Exception as e:
        print("❌ เกิดข้อผิดพลาดขณะ insert:", e)
else:
    print("⚠️ ไม่มีข้อมูลใหม่ให้เพิ่ม (ทั้งหมดซ้ำ)")

📌 CSV มีทั้งหมด 100379 แถว
🔍 พบข้อมูลซ้ำใน MongoDB จำนวน 0 รายการ
✨ จะทำการ insert เฉพาะข้อมูลใหม่จำนวน 100379 รายการ
✅ Insert สำเร็จ!


In [ ]:
import pandas as pd
from pymongo import MongoClient
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
connection_string = MONGO_URI
client = MongoClient(connection_string)

db = client["my_project"]
collection = db["traffic_clean"]
cursor = collection.find({})

# 4. แปลงข้อมูลเป็น Pandas DataFrame
df = pd.DataFrame(list(cursor))

if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "traffy-fondue"
OUTPUT_PATH = PROCESSED_DIR/"traffy_fondue_bangkok_processed.csv"

# เซฟเป็น CSV ด้วย pandas (ไม่ผ่าน Hadoop)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("✅ Saved CSV to:", OUTPUT_PATH)


✅ Saved CSV to: C:\Year_2\DSDE\dsdengdeng-project-dsde\data\processed\traffy-fondue\traffy_fondue_bangkok_processed.csv
